# 🚬 흡연 분류 V10 - V3 복원 + 안정성 강화

## 핵심 전략
- ✅ **V3 피처 엔지니어링 복원** (50+개 피처)
- ✅ **RandomForest 복원** (V3에서 효과적)
- ✅ **ExtraTrees/Stacking 제거** (V8에서 점수 하락 원인)
- ✅ **5시드 × 5폴드 OOF 앙상블**
- ✅ **가중치 최적화 + 임계값 0.005 단위 탐색**

---

## STEP 0: 환경 설정

In [ ]:
!pip install -q xgboost lightgbm catboost

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
set_seed(42)

# 경로 설정
base_path = '/content/drive/MyDrive/AI_Projects/smoking_hackathon/'
train_path = base_path + 'data/train.csv'
test_path = base_path + 'data/test.csv'
submission_path = base_path + 'data/sample_submission.csv'
result_path = base_path + 'results/'

print("✅ STEP 0: 환경 설정 완료!")

## STEP 1: 데이터 로드 + 기본 검증

In [ ]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print("=" * 60)
print("📊 STEP 1: 데이터 로드 완료")
print("=" * 60)
print(f"Train: {train.shape}")
print(f"Test: {test.shape}")
print(f"\n컬럼: {train.columns.tolist()}")

# 타깃 분포
print(f"\n🎯 타깃 분포:")
print(train['label'].value_counts())
smoking_ratio = train['label'].mean()
print(f"\n흡연자 비율: {smoking_ratio*100:.2f}%")

## STEP 2: 한글 컬럼 매핑

In [ ]:
def map_korean_columns(df):
    """
    한글 컬럼명을 영어로 매핑
    """
    df = df.copy()
    col_map = {}
    
    for col in df.columns:
        c = col.lower()
        if 'id' in c:
            col_map[col] = 'id'
        elif '나이' in col or 'age' in c:
            col_map[col] = 'age'
        elif '키' in col and ('cm' in col or '(' in col):
            col_map[col] = 'height'
        elif '몸무게' in col or '체중' in col:
            col_map[col] = 'weight'
        elif 'bmi' in c:
            col_map[col] = 'bmi'
        elif '시력' in col and '좌' in col:
            col_map[col] = 'eyesight_left'
        elif '시력' in col and '우' in col:
            col_map[col] = 'eyesight_right'
        elif '청력' in col and '좌' in col:
            col_map[col] = 'hearing_left'
        elif '청력' in col and '우' in col:
            col_map[col] = 'hearing_right'
        elif '충치' in col:
            col_map[col] = 'cavity'
        elif '수축기' in col or ('혈압' in col and '수축' in col):
            col_map[col] = 'systolic'
        elif '이완기' in col or ('혈압' in col and '이완' in col):
            col_map[col] = 'diastolic'
        elif '혈당' in col or '공복' in col:
            col_map[col] = 'fasting_blood_sugar'
        elif '중성' in col:
            col_map[col] = 'triglyceride'
        elif '크레아티닌' in col or '크레' in col:
            col_map[col] = 'serum_creatinine'
        elif '콜레스테롤' in col and '고밀도' not in col and '저밀도' not in col:
            col_map[col] = 'cholesterol'
        elif '고밀도' in col:
            col_map[col] = 'hdl'
        elif '저밀도' in col:
            col_map[col] = 'ldl'
        elif '헤모글로빈' in col:
            col_map[col] = 'hemoglobin'
        elif '요단백' in col or ('단백' in col and '지단백' not in col):
            col_map[col] = 'urine_protein'
        elif 'ast' in c:
            col_map[col] = 'ast'
        elif 'alt' in c:
            col_map[col] = 'alt'
        elif '감마' in col or 'gtp' in c or 'γ' in col:
            col_map[col] = 'gtp'
        elif 'label' in c:
            col_map[col] = 'label'
        else:
            col_map[col] = col
    
    return df.rename(columns=col_map)

# 컬럼 매핑 적용
train_mapped = map_korean_columns(train)
test_mapped = map_korean_columns(test)

print("✅ STEP 2: 한글 컬럼 매핑 완료")
print(f"매핑 후 컬럼: {train_mapped.columns.tolist()}")

## STEP 3: 피처 엔지니어링 (V3 스타일 복원)

In [ ]:
def create_features_v10(df):
    """
    V3 스타일 피처 엔지니어링 - 50+개 피처 생성
    """
    df = df.copy()
    cols = df.columns.tolist()
    
    # ============================================
    # 1. 콜레스테롤 관련 (흡연자 HDL↓, LDL↑)
    # ============================================
    if 'hdl' in cols and 'ldl' in cols:
        df['HDL_LDL_ratio'] = df['hdl'] / (df['ldl'] + 1)
        df['LDL_HDL_ratio'] = df['ldl'] / (df['hdl'] + 1)
        df['LDL_HDL_diff'] = df['ldl'] - df['hdl']
    
    if 'cholesterol' in cols and 'hdl' in cols:
        df['HDL_Chol_ratio'] = df['hdl'] / (df['cholesterol'] + 1)
        df['NonHDL_Chol'] = df['cholesterol'] - df['hdl']
        df['Atherogenic_idx'] = (df['cholesterol'] - df['hdl']) / (df['hdl'] + 1)
    
    if 'triglyceride' in cols and 'hdl' in cols:
        df['TG_HDL_ratio'] = df['triglyceride'] / (df['hdl'] + 1)
    
    if 'triglyceride' in cols and 'cholesterol' in cols:
        df['TG_Chol_ratio'] = df['triglyceride'] / (df['cholesterol'] + 1)
    
    # ============================================
    # 2. 간 기능 (흡연자 GTP↑)
    # ============================================
    if 'gtp' in cols:
        df['GTP_log'] = np.log1p(df['gtp'])
        df['GTP_sqrt'] = np.sqrt(df['gtp'])
        df['GTP_sq'] = df['gtp'] ** 2
        df['GTP_high'] = (df['gtp'] > 50).astype(int)
    
    if 'ast' in cols and 'alt' in cols:
        df['AST_ALT_ratio'] = df['ast'] / (df['alt'] + 1)
        df['Liver_sum'] = df['ast'] + df['alt']
        df['Liver_diff'] = abs(df['ast'] - df['alt'])
    
    if 'gtp' in cols and 'ast' in cols:
        df['GTP_AST_ratio'] = df['gtp'] / (df['ast'] + 1)
    
    # ============================================
    # 3. 헤모글로빈 (흡연자 높음 - 산소 보상)
    # ============================================
    if 'hemoglobin' in cols:
        df['Hemo_sq'] = df['hemoglobin'] ** 2
        df['Hemo_log'] = np.log1p(df['hemoglobin'])
        df['Hemo_high'] = (df['hemoglobin'] > 15).astype(int)
        df['Hemo_vhigh'] = (df['hemoglobin'] > 16).astype(int)
    
    if 'hemoglobin' in cols and 'gtp' in cols:
        df['Hemo_x_GTP'] = df['hemoglobin'] * df['gtp']
    
    if 'hemoglobin' in cols and 'triglyceride' in cols:
        df['Hemo_x_TG'] = df['hemoglobin'] * df['triglyceride']
    
    # ============================================
    # 4. 혈압 관련
    # ============================================
    if 'systolic' in cols and 'diastolic' in cols:
        df['Pulse_pressure'] = df['systolic'] - df['diastolic']
        df['MAP'] = df['diastolic'] + (df['systolic'] - df['diastolic']) / 3
        df['BP_ratio'] = df['systolic'] / (df['diastolic'] + 1)
        df['BP_high'] = ((df['systolic'] > 130) | (df['diastolic'] > 85)).astype(int)
    
    # ============================================
    # 5. 체형 관련
    # ============================================
    if 'height' in cols and 'weight' in cols:
        height_m = df['height'] / 100
        df['BMI_calc'] = df['weight'] / (height_m ** 2 + 0.01)
        df['BMI_sq'] = df['BMI_calc'] ** 2
    
    if 'bmi' in cols:
        df['BMI_group'] = pd.cut(df['bmi'], bins=[0, 18.5, 23, 25, 100], labels=[0, 1, 2, 3]).astype(float)
    
    # ============================================
    # 6. 시력
    # ============================================
    eye_cols = [c for c in cols if 'eyesight' in c]
    if len(eye_cols) >= 2:
        df['Eyesight_avg'] = df[eye_cols].mean(axis=1)
        df['Eyesight_diff'] = abs(df[eye_cols[0]] - df[eye_cols[1]])
        df['Eyesight_min'] = df[eye_cols].min(axis=1)
    
    # ============================================
    # 7. 나이 관련
    # ============================================
    if 'age' in cols:
        df['Age_sq'] = df['age'] ** 2
        df['Age_group'] = pd.cut(df['age'], bins=[0, 30, 40, 50, 60, 100], labels=[0, 1, 2, 3, 4]).astype(float)
        
        if 'hemoglobin' in cols:
            df['Age_x_Hemo'] = df['age'] * df['hemoglobin']
        if 'gtp' in cols:
            df['Age_x_GTP'] = df['age'] * df['gtp']
        if 'triglyceride' in cols:
            df['Age_x_TG'] = df['age'] * df['triglyceride']
        if 'systolic' in cols:
            df['Age_x_SBP'] = df['age'] * df['systolic']
        if 'cholesterol' in cols:
            df['Age_x_Chol'] = df['age'] * df['cholesterol']
    
    # ============================================
    # 8. 혈당 관련
    # ============================================
    if 'fasting_blood_sugar' in cols:
        df['FBS_log'] = np.log1p(df['fasting_blood_sugar'])
        df['FBS_high'] = (df['fasting_blood_sugar'] > 100).astype(int)
        df['FBS_diabetes'] = (df['fasting_blood_sugar'] > 126).astype(int)
    
    # ============================================
    # 9. 중성지방 관련
    # ============================================
    if 'triglyceride' in cols:
        df['TG_log'] = np.log1p(df['triglyceride'])
        df['TG_sq'] = df['triglyceride'] ** 2
        df['TG_high'] = (df['triglyceride'] > 150).astype(int)
    
    # ============================================
    # 10. 크레아티닌
    # ============================================
    if 'serum_creatinine' in cols:
        df['Creat_log'] = np.log1p(df['serum_creatinine'])
        df['Creat_high'] = (df['serum_creatinine'] > 1.2).astype(int)
    
    # ============================================
    # 11. 종합 건강 점수
    # ============================================
    health_cols = [c for c in ['systolic', 'diastolic', 'hemoglobin', 'triglyceride', 
                               'cholesterol', 'hdl', 'ldl', 'gtp'] if c in cols]
    if len(health_cols) >= 3:
        df['Health_mean'] = df[health_cols].mean(axis=1)
        df['Health_std'] = df[health_cols].std(axis=1)
        df['Health_max'] = df[health_cols].max(axis=1)
        df['Health_min'] = df[health_cols].min(axis=1)
        df['Health_range'] = df['Health_max'] - df['Health_min']
    
    # ============================================
    # 12. 흡연 위험 점수
    # ============================================
    risk_score = 0
    if 'Hemo_high' in df.columns:
        risk_score = risk_score + df['Hemo_high']
    if 'GTP_high' in df.columns:
        risk_score = risk_score + df['GTP_high']
    if 'TG_high' in df.columns:
        risk_score = risk_score + df['TG_high']
    if 'BP_high' in df.columns:
        risk_score = risk_score + df['BP_high']
    df['Smoking_risk_score'] = risk_score
    
    # 결측치/무한값 처리
    df = df.fillna(0)
    df = df.replace([np.inf, -np.inf], 0)
    
    return df

# 피처 엔지니어링 적용
print("🔧 STEP 3: 피처 엔지니어링 적용 중...")

# ID 제거
train_df = train_mapped.drop(['id'], axis=1, errors='ignore')
test_df = test_mapped.drop(['id'], axis=1, errors='ignore')

original_cols = len(train_df.columns) - 1  # label 제외

train_fe = create_features_v10(train_df)
test_fe = create_features_v10(test_df)

print(f"\n✅ STEP 3: 피처 엔지니어링 완료!")
print(f"   원본: {original_cols}개 → 새로운: {train_fe.shape[1]-1}개")
print(f"   추가된 피처: {train_fe.shape[1]-1-original_cols}개")

## STEP 4: 전처리 (이상치 + 스케일링)

In [ ]:
# X, y 분리
X = train_fe.drop('label', axis=1)
y = train_fe['label']
X_test = test_fe.drop('label', axis=1, errors='ignore')

# 컬럼 순서 맞추기
X_test = X_test[X.columns]

print(f"X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}")

# 클래스 비율
n_neg = (y == 0).sum()
n_pos = (y == 1).sum()
scale_pos = n_neg / n_pos
print(f"\n클래스 분포: 비흡연={n_neg}, 흡연={n_pos}")
print(f"scale_pos_weight: {scale_pos:.2f}")

In [ ]:
# 이상치 클리핑 (IQR 3.0)
def clip_outliers(df, multiplier=3.0):
    df = df.copy()
    for col in df.columns:
        if df[col].dtype in ['float64', 'int64', 'float32', 'int32']:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            df[col] = df[col].clip(lower=Q1 - multiplier * IQR, upper=Q3 + multiplier * IQR)
    return df

X_clipped = clip_outliers(X)
X_test_clipped = clip_outliers(X_test)

print("✅ 이상치 클리핑 완료 (IQR 3.0)")

In [ ]:
# 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clipped)
X_test_scaled = scaler.transform(X_test_clipped)

# DataFrame 변환 (CatBoost용)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)

print(f"✅ STEP 4: 전처리 완료!")
print(f"   X_scaled: {X_scaled.shape}")
print(f"   X_test_scaled: {X_test_scaled.shape}")

## STEP 5: 하이퍼파라미터 튜닝 (n_iter=100)

In [ ]:
print("=" * 60)
print("🔧 STEP 5: 하이퍼파라미터 튜닝 (Accuracy 기준)")
print("=" * 60)

In [ ]:
# [1/4] XGBoost 튜닝
print("\n[1/4] XGBoost 튜닝 중...")

xgb_params = {
    'n_estimators': [300, 500, 700, 1000],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'min_child_weight': [1, 3, 5, 7],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'gamma': [0, 0.1, 0.2],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [1, 2, 5],
    'scale_pos_weight': [1, scale_pos]
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, verbosity=0, use_label_encoder=False, eval_metric='logloss'),
    xgb_params, n_iter=100, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
xgb_search.fit(X_scaled, y)
best_xgb = xgb_search.best_params_
print(f"\n✅ XGBoost 최고 Accuracy: {xgb_search.best_score_:.5f}")

In [ ]:
# [2/4] LightGBM 튜닝
print("\n[2/4] LightGBM 튜닝 중...")

lgb_params = {
    'n_estimators': [300, 500, 700, 1000],
    'max_depth': [3, 5, 7, -1],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'num_leaves': [15, 31, 63],
    'min_child_samples': [10, 20, 30, 50],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0, 0.1, 0.5],
    'class_weight': ['balanced', None]
}

lgb_search = RandomizedSearchCV(
    LGBMClassifier(random_state=42, verbose=-1),
    lgb_params, n_iter=100, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
lgb_search.fit(X_scaled, y)
best_lgb = lgb_search.best_params_
print(f"\n✅ LightGBM 최고 Accuracy: {lgb_search.best_score_:.5f}")

In [ ]:
# [3/4] CatBoost 튜닝
print("\n[3/4] CatBoost 튜닝 중...")

cat_params = {
    'n_estimators': [300, 500, 700, 1000],
    'max_depth': [4, 5, 6, 7],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'l2_leaf_reg': [1, 3, 5, 7],
    'auto_class_weights': ['Balanced', None]
}

cat_search = RandomizedSearchCV(
    CatBoostClassifier(random_state=42, verbose=0),
    cat_params, n_iter=60, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
cat_search.fit(X_scaled_df, y)
best_cat = cat_search.best_params_
print(f"\n✅ CatBoost 최고 Accuracy: {cat_search.best_score_:.5f}")

In [ ]:
# [4/4] RandomForest 튜닝 (V3에서 효과적)
print("\n[4/4] RandomForest 튜닝 중...")

rf_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', 'balanced_subsample', None]
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_params, n_iter=60, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
rf_search.fit(X_scaled, y)
best_rf = rf_search.best_params_
print(f"\n✅ RandomForest 최고 Accuracy: {rf_search.best_score_:.5f}")

In [ ]:
print("\n" + "=" * 60)
print("📊 튜닝 결과 요약 (Accuracy 기준)")
print("=" * 60)
print(f"XGBoost:      {xgb_search.best_score_:.5f}")
print(f"LightGBM:     {lgb_search.best_score_:.5f}")
print(f"CatBoost:     {cat_search.best_score_:.5f}")
print(f"RandomForest: {rf_search.best_score_:.5f}")

## STEP 6: 5시드 × 5폴드 OOF 앙상블

In [ ]:
print("=" * 60)
print("🎯 STEP 6: 5시드 × 5폴드 OOF 앙상블")
print("=" * 60)

SEEDS = [42, 123, 456, 789, 2024]
N_SPLITS = 5

# 시드별 OOF/Test 저장
all_oof_xgb = []
all_oof_lgb = []
all_oof_cat = []
all_oof_rf = []

all_test_xgb = []
all_test_lgb = []
all_test_cat = []
all_test_rf = []

In [ ]:
for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*50}")
    print(f"🔄 시드 {seed} ({seed_idx+1}/{len(SEEDS)})")
    print(f"{'='*50}")
    
    kfold = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    
    oof_xgb = np.zeros(len(X_scaled))
    oof_lgb = np.zeros(len(X_scaled))
    oof_cat = np.zeros(len(X_scaled))
    oof_rf = np.zeros(len(X_scaled))
    
    test_xgb = np.zeros(len(X_test_scaled))
    test_lgb = np.zeros(len(X_test_scaled))
    test_cat = np.zeros(len(X_test_scaled))
    test_rf = np.zeros(len(X_test_scaled))
    
    for fold, (tr_idx, va_idx) in enumerate(kfold.split(X_scaled, y)):
        print(f"  Fold {fold+1}/{N_SPLITS}...", end=" ")
        
        X_tr, X_va = X_scaled[tr_idx], X_scaled[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        
        # DataFrame 버전 (CatBoost용)
        X_tr_df = X_scaled_df.iloc[tr_idx]
        X_va_df = X_scaled_df.iloc[va_idx]
        
        # XGBoost
        xgb_m = XGBClassifier(**best_xgb, random_state=seed, verbosity=0, use_label_encoder=False, eval_metric='logloss')
        xgb_m.fit(X_tr, y_tr)
        oof_xgb[va_idx] = xgb_m.predict_proba(X_va)[:, 1]
        test_xgb += xgb_m.predict_proba(X_test_scaled)[:, 1] / N_SPLITS
        
        # LightGBM
        lgb_m = LGBMClassifier(**best_lgb, random_state=seed, verbose=-1)
        lgb_m.fit(X_tr, y_tr)
        oof_lgb[va_idx] = lgb_m.predict_proba(X_va)[:, 1]
        test_lgb += lgb_m.predict_proba(X_test_scaled)[:, 1] / N_SPLITS
        
        # CatBoost
        cat_m = CatBoostClassifier(**best_cat, random_state=seed, verbose=0)
        cat_m.fit(X_tr_df, y_tr)
        oof_cat[va_idx] = cat_m.predict_proba(X_va_df)[:, 1]
        test_cat += cat_m.predict_proba(X_test_scaled_df)[:, 1] / N_SPLITS
        
        # RandomForest
        rf_m = RandomForestClassifier(**best_rf, random_state=seed, n_jobs=-1)
        rf_m.fit(X_tr, y_tr)
        oof_rf[va_idx] = rf_m.predict_proba(X_va)[:, 1]
        test_rf += rf_m.predict_proba(X_test_scaled)[:, 1] / N_SPLITS
        
        print("✓")
    
    # 시드별 저장
    all_oof_xgb.append(oof_xgb)
    all_oof_lgb.append(oof_lgb)
    all_oof_cat.append(oof_cat)
    all_oof_rf.append(oof_rf)
    
    all_test_xgb.append(test_xgb)
    all_test_lgb.append(test_lgb)
    all_test_cat.append(test_cat)
    all_test_rf.append(test_rf)
    
    # 시드별 성능
    seed_acc_xgb = accuracy_score(y, (oof_xgb >= 0.5).astype(int))
    seed_acc_lgb = accuracy_score(y, (oof_lgb >= 0.5).astype(int))
    seed_acc_cat = accuracy_score(y, (oof_cat >= 0.5).astype(int))
    seed_acc_rf = accuracy_score(y, (oof_rf >= 0.5).astype(int))
    print(f"  → XGB: {seed_acc_xgb:.4f}, LGB: {seed_acc_lgb:.4f}, CAT: {seed_acc_cat:.4f}, RF: {seed_acc_rf:.4f}")

print("\n✅ STEP 6: 5시드 × 5폴드 학습 완료!")

In [ ]:
# 5시드 평균
oof_xgb_avg = np.mean(all_oof_xgb, axis=0)
oof_lgb_avg = np.mean(all_oof_lgb, axis=0)
oof_cat_avg = np.mean(all_oof_cat, axis=0)
oof_rf_avg = np.mean(all_oof_rf, axis=0)

test_xgb_avg = np.mean(all_test_xgb, axis=0)
test_lgb_avg = np.mean(all_test_lgb, axis=0)
test_cat_avg = np.mean(all_test_cat, axis=0)
test_rf_avg = np.mean(all_test_rf, axis=0)

print("📊 5시드 평균 OOF 성능 (threshold=0.5):")
print(f"   XGBoost:      {accuracy_score(y, (oof_xgb_avg >= 0.5).astype(int)):.5f}")
print(f"   LightGBM:     {accuracy_score(y, (oof_lgb_avg >= 0.5).astype(int)):.5f}")
print(f"   CatBoost:     {accuracy_score(y, (oof_cat_avg >= 0.5).astype(int)):.5f}")
print(f"   RandomForest: {accuracy_score(y, (oof_rf_avg >= 0.5).astype(int)):.5f}")

## STEP 7: 최적 가중치 탐색

In [ ]:
print("=" * 60)
print("🔍 STEP 7: 최적 가중치 탐색 (0.05 단위)")
print("=" * 60)

best_weight_score = 0
best_weights = None

for w1 in np.arange(0.1, 0.6, 0.05):  # XGB
    for w2 in np.arange(0.1, 0.6, 0.05):  # LGB
        for w3 in np.arange(0.1, 0.6, 0.05):  # CAT
            w4 = round(1 - w1 - w2 - w3, 2)  # RF
            if w4 >= 0.05:
                prob = w1*oof_xgb_avg + w2*oof_lgb_avg + w3*oof_cat_avg + w4*oof_rf_avg
                pred_label = (prob >= 0.5).astype(int)
                acc = accuracy_score(y, pred_label)
                if acc > best_weight_score:
                    best_weight_score = acc
                    best_weights = (w1, w2, w3, w4)

w_xgb, w_lgb, w_cat, w_rf = best_weights

print(f"\n🏆 최적 가중치:")
print(f"   XGBoost:      {w_xgb:.2f}")
print(f"   LightGBM:     {w_lgb:.2f}")
print(f"   CatBoost:     {w_cat:.2f}")
print(f"   RandomForest: {w_rf:.2f}")
print(f"\n   가중합 OOF Accuracy: {best_weight_score:.5f}")

In [ ]:
# 가중 평균 적용
oof_ensemble = w_xgb*oof_xgb_avg + w_lgb*oof_lgb_avg + w_cat*oof_cat_avg + w_rf*oof_rf_avg
test_ensemble = w_xgb*test_xgb_avg + w_lgb*test_lgb_avg + w_cat*test_cat_avg + w_rf*test_rf_avg

print(f"앙상블 OOF 확률 범위: [{oof_ensemble.min():.4f}, {oof_ensemble.max():.4f}]")

## STEP 8: 최적 임계값 탐색 (0.005 단위)

In [ ]:
print("=" * 60)
print("🔍 STEP 8: 최적 임계값 탐색 (0.005 단위)")
print("=" * 60)

best_th = 0.5
best_acc = 0
best_f1 = 0
results = []

for th in np.arange(0.35, 0.65, 0.005):
    pred = (oof_ensemble >= th).astype(int)
    acc = accuracy_score(y, pred)
    f1 = f1_score(y, pred)
    results.append({'threshold': round(th, 3), 'accuracy': acc, 'f1': f1})
    if acc > best_acc:
        best_acc = acc
        best_f1 = f1
        best_th = round(th, 3)

results_df = pd.DataFrame(results)
print("\n상위 15개 임계값:")
print(results_df.nlargest(15, 'accuracy').to_string(index=False))

print(f"\n🏆 최적 임계값: {best_th:.3f}")
print(f"   OOF Accuracy: {best_acc:.5f}")
print(f"   OOF F1-Score: {best_f1:.5f}")

In [ ]:
# 혼동 행렬
val_pred_final = (oof_ensemble >= best_th).astype(int)

print("\n📊 OOF 혼동 행렬:")
print(confusion_matrix(y, val_pred_final))
print("\n📊 Classification Report:")
print(classification_report(y, val_pred_final, target_names=['비흡연(0)', '흡연(1)']))

## STEP 9: 제출 파일 생성 (5개)

In [ ]:
print("=" * 60)
print("📝 STEP 9: 제출 파일 생성")
print("=" * 60)

# 최적 임계값 ± 0.02, ± 0.04
thresholds = [
    round(best_th - 0.04, 3),
    round(best_th - 0.02, 3),
    round(best_th, 3),
    round(best_th + 0.02, 3),
    round(best_th + 0.04, 3)
]

file_paths = []

for th in thresholds:
    pred = (test_ensemble >= th).astype(int)
    oof_pred = (oof_ensemble >= th).astype(int)
    oof_acc = accuracy_score(y, oof_pred)
    
    sub = submission.copy()
    sub['label'] = pred
    sub['label'] = sub['label'].astype(int)
    
    th_str = str(int(th * 1000)).zfill(3)
    filename = f'submission_v10_t{th_str}.csv'
    filepath = result_path + filename
    sub.to_csv(filepath, index=False)
    file_paths.append(filepath)
    
    n_smoking = (pred == 1).sum()
    pct = n_smoking / len(pred) * 100
    
    marker = "⭐" if th == best_th else "  "
    print(f"\n{marker} {filename}")
    print(f"   임계값: {th:.3f}")
    print(f"   OOF Accuracy: {oof_acc:.5f}")
    print(f"   예측: 비흡연={len(pred)-n_smoking} ({100-pct:.1f}%), 흡연={n_smoking} ({pct:.1f}%)")

print(f"\n✅ {len(thresholds)}개 제출 파일 생성 완료!")

In [ ]:
# 검증
print("\n🔍 제출 파일 검증:")
for fp in file_paths:
    df = pd.read_csv(fp)
    fn = fp.split('/')[-1]
    valid = df['label'].dtype in ['int64', 'int32'] and set(df['label'].unique()).issubset({0, 1})
    print(f"   {'✅' if valid else '❌'} {fn}: {df.shape}, dtype={df['label'].dtype}")

## STEP 10: 다운로드 + 요약

In [ ]:
from google.colab import files

# 최적 임계값 파일 다운로드
best_file = result_path + f'submission_v10_t{str(int(best_th*1000)).zfill(3)}.csv'
files.download(best_file)

print("\n" + "=" * 60)
print("🎉 V10 완료!")
print("=" * 60)
print(f"\n📊 최종 결과:")
print(f"   모델: XGBoost + LightGBM + CatBoost + RandomForest")
print(f"   가중치: XGB={w_xgb:.2f}, LGB={w_lgb:.2f}, CAT={w_cat:.2f}, RF={w_rf:.2f}")
print(f"   시드: {SEEDS}")
print(f"   피처 수: {X_scaled.shape[1]}개")
print(f"   최적 임계값: {best_th:.3f}")
print(f"   OOF Accuracy: {best_acc:.5f}")
print(f"\n📁 생성된 파일:")
for fp in file_paths:
    fn = fp.split('/')[-1]
    marker = "👉" if f't{str(int(best_th*1000)).zfill(3)}' in fn else "  "
    print(f"   {marker} {fn}")
print(f"\n🚀 ⭐ 파일 먼저 제출!")

In [ ]:
# 다른 파일도 다운로드
print("📥 추가 파일 다운로드:")
for fp in file_paths:
    if fp != best_file:
        files.download(fp)
        print(f"   ✅ {fp.split('/')[-1]}")